In [1]:
import pandas as pd
import gc

In [ ]:
df2 = pd.read_pickle('../data/df2.pkl')

In [ ]:
df2.head()

In [ ]:
df2.info() # (49333714, 25)

In [ ]:
# select representative date
df2 = df2[df2['ENTRY_DT'] == '2025-02-11']

In [ ]:
gc.collect() # free up storage immediately to delete overwritten files - the big df2 at the start
print(df2.shape) # (3846784, 25) for 11 feb

# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: Sort by CRD_NUM and ENTRY_TM, flag rows where the next row belongs to a different CRD_NUM.
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


In [ ]:
# Drop rows with missing critical data
critical_cols = [
    'ENTRY_TM', 'EXIT_TM',
    'ORIG_LOC_ID_NUM', 'DEST_LOC_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df2)
df2 = df2.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df2)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

In [ ]:
# Binary criteria
# Sort first
df2 = df2.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)

# Shift to get next row's CRD_NUM
df2['next_CRD_NUM'] = df2['CRD_NUM'].shift(-1)

# Last stage = next row belongs to a different commuter
df2['is_last_stage'] = df2['CRD_NUM'] != df2['next_CRD_NUM'] # This tells us the last ride for the specific commuter

print(df2['is_last_stage'].value_counts())
print(f"\nUnique commuters: {df2['CRD_NUM'].nunique():,}") # Check that no. if True in is_last_stage == No. of unique commuters

In [ ]:
# Shift needed columns
df2['next_ORIG_LOC_ID_NUM'] = df2['ORIG_LOC_ID_NUM'].shift(-1)
df2['next_BUS_SVC_NUM']     = df2['BUS_SVC_NUM'].shift(-1)

# Flag same service return
df2['same_service_return'] = (
    (df2['CRD_NUM'] == df2['next_CRD_NUM']) &
    (df2['BUS_SVC_NUM'] == df2['next_BUS_SVC_NUM']) &
    (df2['DEST_LOC_ID_NUM'] == df2['next_ORIG_LOC_ID_NUM'])
)

print(df2['same_service_return'].value_counts())

#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
- Allowance is given by **LTA's current window**:
    - Bus↔Bus or Bus↔Train transfers: 45 min maximum gap
    - Train↔Train transfers: 15 min maximum gap
    - Same bus service number: not a valid transfer (already handled in binary)
    - Max 5 transfers per journey
    - Overall journey duration (first to last boarding): max 2 hours


# PAUSE HERE BEFORE RUNNING BELOW. 

### Is this the correct temporal criteria?

* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

In [ ]:
# Shift needed columns 
df2['next_ENTRY_TM']       = df2['ENTRY_TM'].shift(-1)
df2['next_TRNSPT_MODE_CD'] = df2['TRNSPT_MODE_CD'].shift(-1)

#  Compute time gap
df2['time_gap_mins'] = (
    df2['next_ENTRY_TM'] - df2['EXIT_TM']
).dt.total_seconds() / 60

# Assign threshold by transfer type 
# RTS-RTS (2-2) → 15 mins, everything else → 45 mins
df2['time_threshold'] = df2.apply(lambda row:
    15 if (row['TRNSPT_MODE_CD'] == 2 and row['next_TRNSPT_MODE_CD'] == 2)
    else 45, axis=1
)

# Flag if gap exceeds threshold
df2['exceeds_time_allowance'] = (
    (df2['CRD_NUM'] == df2['next_CRD_NUM']) &
    (df2['time_gap_mins'] > df2['time_threshold'])
)

# Flag if overall journey exceeds 2 hours 
df2['journey_start_TM'] = df2.groupby('CRD_NUM')['ENTRY_TM'].transform('min')
df2['journey_duration_mins'] = (
    df2['ENTRY_TM'] - df2['journey_start_TM']
).dt.total_seconds() / 60
df2['exceeds_2hr'] = df2['journey_duration_mins'] > 120

# Flag if transfer count exceeds 5
df2['transfer_count'] = df2.groupby('CRD_NUM').cumcount()
df2['exceeds_5_transfers'] = df2['transfer_count'] > 5

# Combined temporal flag 
df2['temporal_new_journey'] = (
    df2['exceeds_time_allowance'] |
    df2['exceeds_2hr'] |
    df2['exceeds_5_transfers']
)

print(df2['exceeds_time_allowance'].value_counts())
print(df2['exceeds_2hr'].value_counts())
print(df2['exceeds_5_transfers'].value_counts())
print(f"\nTotal temporal new journey flags: {df2['temporal_new_journey'].sum():,}")

#### 3. Spatial Criteria:
-  The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- Implementation: 
    - Use original clean df to access logtitude and latitude columns
    - Sort the dataset by card number (CRD_NUM) and timestamp (ENTRY_TM) to get consecutive stages for each rider.
    - For each ride, we look at the next stage’s boarding location (latitude and longitude). Rows without a next stage are ignored because there is nothing to compare. The person finished all their trips for the day. 
    - We create a boolean mask to select only rows that have a next stage (NEXT_ENTRY_LAT is not NaN). This helps us to ignore observations without next stages
    - Use Haversine formula to compute the shortest straight-line distance between DEST_LATITUDE & DEST_LONGTITUDE (alighting stop) and the NEXT_ENTRY_LONG & NEXT_ENTRY_LAT (the next entry point). Vectorisation is used because row-by-row computation was too slow. 
    - Allow a maximum transfer distance of 500 metres (we can change this)
    - If the distance exceeds a maximum allowed transfer distance (0.5 km), the row is flagged as spatial_new_journey = True. This indicates that the next ride is likely a new journey.

In [ ]:
df3 = pd.read_pickle('../data/df2.pkl')

In [ ]:
df3.head()

In [ ]:
df3 = df3.sort_values(by=['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)

In [ ]:
# Shift boarding coordinates to get "next stage" per rider
df3['NEXT_ENTRY_LAT'] = df3.groupby('CRD_NUM')['ORIG_LATITUDE'].shift(-1)
df3['NEXT_ENTRY_LON'] = df3.groupby('CRD_NUM')['ORIG_LONGITUDE'].shift(-1)

In [ ]:
import numpy as np
#use vectorized function so it runs much faster than row by row
def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371  # Earth radius in km
    return c * r

In [ ]:
#compute distance to next stage if it exists
mask = df3['NEXT_ENTRY_LAT'].notna()  # skip last stage (ie. if there is no next tap in) 
df3.loc[mask, 'distance_to_next_km'] = haversine_vectorized(
    df3.loc[mask, 'DEST_LATITUDE'].values,
    df3.loc[mask, 'DEST_LONGITUDE'].values,
    df3.loc[mask, 'NEXT_ENTRY_LAT'].values,
    df3.loc[mask, 'NEXT_ENTRY_LON'].values
)

In [ ]:
#flag spatial transfers
# set max reasonable walking transfer distance = 0.5 km 
# we can change this threshold
max_transfer_distance_km = 0.5

#initialise as False. Only mark as True if distance to next stage exceeds threshold
df3['spatial_new_journey'] = False
df3.loc[mask, 'spatial_new_journey'] = df3.loc[mask, 'distance_to_next_km'] > max_transfer_distance_km

In [ ]:
df3['spatial_new_journey'].value_counts()

# Internal Validity Check

In [2]:
df5 = pd.read_pickle('../data/df5.pkl')

In [3]:
df5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33956202 entries, 0 to 33956201
Data columns (total 23 columns):
 #   Column              Dtype         
---  ------              -----         
 0   CRD_NUM             int64         
 1   JRNY_DEST_ID_NUM    float64       
 2   JRNY_START_DT       datetime64[ns]
 3   JRNY_START_TM       datetime64[ns]
 4   JRNY_END_DT         datetime64[ns]
 5   JRNY_END_TM         datetime64[ns]
 6   JRNY_ORIG_ID_NUM    float64       
 7   JRNY_DIST_KM_CNT    float64       
 8   JRNY_FARE_AMT       float64       
 9   JRNY_ID_NUM         int64         
 10  JRNY_TM_MIN_CNT     float64       
 11  PATRON_CATG_ID_NUM  int64         
 12  TRNSPT_MODE_CD      int64         
 13  DEST_STATION_NAME   object        
 14  DEST_MRK_ID_NUM     float64       
 15  DEST_LATITUDE       float64       
 16  DEST_LONGITUDE      float64       
 17  DEST_Travel_Type    object        
 18  ORIG_STATION_NAME   object        
 19  ORIG_MRK_ID_NUM     float64       
 20  

In [ ]:
df4 = df5[df5['JRNY_START_DT'] == '2025-02-11']

#### 1. Binary Criteria
- Drop rows with missing critical data
- 


In [4]:
# Drop rows with missing critical data
critical_cols = [
    'JRNY_START_TM', 'JRNY_END_TM',
    'JRNY_ORIG_ID_NUM', 'JRNY_DEST_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df5)
df5 = df5.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df5)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

MemoryError: Unable to allocate 510. MiB for an array with shape (2, 33448850) and data type object